<a href="https://colab.research.google.com/github/Terheer/Cnn/blob/main/cnnresnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms


from torch.utils.data import DataLoader, random_split, Subset

import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms, datasets
from tqdm import tqdm

In [ ]:
import zipfile

with zipfile.ZipFile('archive (16).zip', 'r') as zip_ref:
    zip_ref.extractall('extracted_folder')

print("Extraction complete!")

Extraction complete!


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
from torchvision import datasets

root_dir = "/content/extracted_folder/Dataset"


full_aug_dataset = datasets.ImageFolder(root=root_dir, transform=train_transform)
full_eval_dataset = datasets.ImageFolder(root=root_dir, transform=eval_transform)

generator = torch.Generator().manual_seed(42)
train_idx, val_idx, test_idx = random_split(
    range(len(full_aug_dataset)),
    [0.7, 0.15, 0.15],
    generator=generator
)

In [ ]:
train_dataset = Subset(full_aug_dataset, train_idx.indices)
val_dataset = Subset(full_eval_dataset, val_idx.indices)
test_dataset = Subset(full_eval_dataset, test_idx.indices)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Classes found: {full_aug_dataset.classes}")
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)} | Test size: {len(test_dataset)}")

Classes found: ['Bus', 'Car', 'Truck', 'motorcycle']
Train size: 280 | Val size: 60 | Test size: 60


In [ ]:
import os
from PIL import Image

#find the first image file in the dataset directory
root_dir = "/content/extracted_folder/Dataset"
found_path = None

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            found_path = os.path.join(root, file)
            break
    if found_path:
        break

if found_path:
    print(f"Loading image from: {found_path}")
    img = Image.open(found_path)
    if img.mode == "P" and "transparency" in img.info:
        img = img.convert("RGBA")
else:
    print("No image files found in the dataset directory.")



Loading image from: /content/extracted_folder/Dataset/Bus/Image_55.jpg


In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes",
    category=UserWarning,
    module="PIL.Image",
)

In [ ]:
import torchvision.models as models

class VehicleCNN(nn.Module):
    def __init__(self, num_classes, fine_tune=True):
        super(VehicleCNN, self).__init__()
        #Loading pre-trained ResNet-50
        self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        #Freezing early layers since dataset is small
        if not fine_tune:
            for param in self.model.parameters():
                param.requires_grad = False

        #Replace the final fully connected layer
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VehicleCNN(num_classes=4).to(device)
print(model)

VehicleCNN(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import copy
from tqdm import tqdm
import torch.optim as optim

# Ensure loss function and optimizer are defined
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Define training configuration
num_epochs = 10
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)

    # Each epoch has a training and validation phase
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Set model to training mode
            dataloader = train_loader
        else:
            model.eval()   # Set model to evaluate mode
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0

        # Iterate over data
        for inputs, labels in tqdm(dataloader, desc=f"{phase.capitalize()} Phase"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass: track history only if in train
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # Backward pass + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)

        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

        # Deep copy the model if it has the best validation accuracy so far
        if phase == 'val':
            scheduler.step(epoch_loss)
            if epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

print(f"\nTraining complete. Best Val Accuracy: {best_acc:.4f}")

# Load best model weights
model.load_state_dict(best_model_wts)


Epoch 1/10
------------------------------


Train Phase: 100%|██████████| 9/9 [02:18<00:00, 15.42s/it]


Train Loss: 0.7235 | Acc: 0.7179


Val Phase: 100%|██████████| 2/2 [00:09<00:00,  4.64s/it]


Val Loss: 0.3042 | Acc: 0.9333

Epoch 2/10
------------------------------


Train Phase: 100%|██████████| 9/9 [02:11<00:00, 14.62s/it]


... (32 outputs left out)

Train Loss: 0.0563 | Acc: 0.9893


Val Phase: 100%|██████████| 2/2 [00:10<00:00,  5.35s/it]


Val Loss: 0.0956 | Acc: 0.9833

Training complete. Best Val Accuracy: 0.9833


<All keys matched successfully>

In [ ]:
torch.save(model.state_dict(), 'train_net.pth')

In [ ]:
from os import pread
correct = 0
total = 0

model.eval()

with torch.no_grad():
  for data in test_loader:
    images, labels = data
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 85 %


In [ ]:
import os
from PIL import Image

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def load_image(image_path):
  img = Image.open(image_path).convert('RGB')
  # Fixed undefined 'new_transform' to 'test_transform'
  img = test_transform(img)
  img = img.unsqueeze(0)
  return img


image_paths = ['piro4d-car-1890494_1920.jpg', 'antriksh-bike-909690_1920.jpg', 'falco-semi-trailer-534577_1920.jpg']
images = []

# Safely load only existing images
for path in image_paths:
  if os.path.exists(path):
    images.append((path, load_image(path).to(device)))
  else:
    print(f"Warning: '{path}' was not found. Please upload it to Colab to make predictions on it.")

# Define class names from dataset
class_names = full_aug_dataset.classes

if images:
  model.eval()
  with torch.no_grad():
    for path, image in images:
      outputs = model(image)
      _, predicted = torch.max(outputs, 1)
      print(f'File: {path} -> Prediction: {class_names[predicted.item()]}')
else:
  print("No valid images to predict. Please upload your images to Colab using the file explorer on the left.")

File: piro4d-car-1890494_1920.jpg -> Prediction: Car
File: antriksh-bike-909690_1920.jpg -> Prediction: motorcycle
File: falco-semi-trailer-534577_1920.jpg -> Prediction: Truck
